In [2]:
import os
os.sys.path.append('/data/scratch/globc/bonassies/workspace/swot_for_flood')

import configparser
from pathlib import Path
from core.swot_project import SwotProject

main_path = "/data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Ohio"

# Download and rasterize notebook

This notebook explain how to use the swot_for_flood library to download and rasterize the SWOT data for a given region and time period. 

## Define a project

This library is designed to work with a project. A project is a directory that contains the configuration file `config.cfg` and the data. The configuration file contains the parameters of the project.

In this notebook, we will create a project named "example_download_rasterize" in the directory "examples". The project will contain the SWOT data for the region of interest and time period defined in the configuration file.

In [3]:
config = configparser.ConfigParser()
config.read(main_path + '/config.cfg')

print(type(config),dict(config['CONFIG']))

<class 'configparser.ConfigParser'> {'project': 'Ohio', 'workspace': '/data/scratch/globc/bonassies/workspace/swot_for_flood/examples', 'data_path': '/data/scratch/globc/bonassies/data', 'aoi': '/data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Ohio/aux_data/aoi_4326.gpkg', 'floodmask_path': '/data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Ohio/aux_data/FloodedArea_32616_v2.gpkg', 'controlmask_path': '/data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Ohio/aux_data/ControlArea_32616_v2.gpkg', 'esa_worldcover_path': '/data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Ohio/aux_data/ESA_WC_Fusion_cut_32616_nrow3646_ncol6003.tif', 'crs': '32616', 'aoi_crs': '4326', 'first_time': '2023-01-01', 'last_time': '2025-05-01', 'nodes': '8', 'download_type': 'PIXC', 'passes': '[481]', 'tile_names_selection': '[[481_220L, 481_221L]]', 'list_dry_dates': '[2023-09-18, 2023-10-29, 2023-12-10, 2024-03-23]', 'list_flood_dates': '[2025-02-20]', 'vari

The config can also be a dictionary

In [4]:
# from pathlib import Path

# param_dict = {
#     'project': 'Greece_EMSR692',
#     'workspace': Path("/data/scratch/globc/bonassies/workspace"),
#     'data_path': Path("/data/scratch/globc/bonassies/data"),
#     'CRS': 'EPSG:32634',
#     'first_time': "2023-09-05",
#     'last_time': "2023-09-20",
#     'aoi': Path("/data/scratch/globc/bonassies/workspace").joinpath('Greece_EMSR692',"aux_data","EMSR692_aois_V2.gpkg"),
#     'aoi_crs': 'EPSG:4326',
#     'passes': [402, 417],
#     'nodes': 8,
#     'do_download': False,
#     'download_type': 'PIXC',
#     'GDAL_NUM_THREADS': 10,
#     'GDAL_CACHEMAX': 160000,
#     'do_make_gpkg': False,
#     'do_make_tiff': False    
# }

Once the config file loaded, we can use it to create a project object. The project object will contain the parameters of the project and the data.

In [5]:
swot_project = SwotProject(config)
print(swot_project)

Data path already exists in /data/scratch/globc/bonassies/data or download is set to False
SWOT data already exists in /data/scratch/globc/bonassies/data/SWOT or download is set to False
SWOT project already exists in /data/scratch/globc/bonassies/data/SWOT/Ohio or download is set to False
Geopackage already exists in /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Ohio/gpkg_combined or make_gpkg is set to False
TIFF path already exists in /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Ohio/rasters or make_tiff is set to False
No automatic download, please use the Downloader object to download the data
Class SWOT_PROJECT():
	project: Ohio
	workspace: /data/scratch/globc/bonassies/workspace/swot_for_flood/examples
	data_path: /data/scratch/globc/bonassies/data
	CRS: 32616
	first_time: 2023-01-01
	last_time: 2025-05-01
	variables: ['sig0', 'coherent_power', 'incidence', 'gamma_tot', 'gamma_SNR', 'height', 'classification', 'bright_land_flag']
	tile_names_

On the first time, you should get an error because the data is not downloaded yet. The project object will download the data and store it in the project directory only if do_download is set to True in the configuration file.

You can manually download the data by calling the download method of the project object.

First we need to search for the data we want to download. We can use the search method of the project object to search for the data. The search method will return a list of the data that match the search criteria.

In [6]:
# swot_project.Downloader.search_PIXCVec(True)
# swot_project.Downloader.download_pool() # if multithreading download
# swot_project.Downloader.search_Nodes(True)
# swot_project.Downloader.download_pool() # if multithreading download

In [ ]:
swot_project.Downloader.search_PIXC(True)

Then we can download the data by calling the download method of the project object. The download method will download the data and store it in the project directory.

In [ ]:
swot_project.Downloader.download_pool() # if multithreading download
# swot_project.Downloader.download_granules() # if single thread download

Finally we can rasterize the data by calling the rasterize method of the project object. The rasterize method will rasterize the data and store it in the project directory.

First we create gpgk file that combine the netcdf files and then we rasterize the data.

In [5]:
swot_project.Rasterizer.make_space = False
swot_project.Rasterizer.find_pixc(True) #True : only pre-treat the ganules on the studied dates (based on dry and flooded dates lists)
print(swot_project.Rasterizer.list_time_select)
print(swot_project.Rasterizer.list_pixc)
print(swot_project.Rasterizer.tile_names_selection)

swot_project.Rasterizer.pixc_to_gpkg()

['20230918', '20231029', '20231210', '20240323', '20250220']
[[PosixPath('/data/scratch/globc/bonassies/data/SWOT/Ohio/SWOT_L2_HR_PIXC_003_481_220L_20230918T031508_20230918T031519_PGC0_01.nc'), PosixPath('/data/scratch/globc/bonassies/data/SWOT/Ohio/SWOT_L2_HR_PIXC_003_481_221L_20230918T031518_20230918T031529_PGC0_01.nc')], [PosixPath('/data/scratch/globc/bonassies/data/SWOT/Ohio/SWOT_L2_HR_PIXC_005_481_220L_20231029T204515_20231029T204526_PGC0_01.nc'), PosixPath('/data/scratch/globc/bonassies/data/SWOT/Ohio/SWOT_L2_HR_PIXC_005_481_221L_20231029T204525_20231029T204536_PGC0_01.nc')], [PosixPath('/data/scratch/globc/bonassies/data/SWOT/Ohio/SWOT_L2_HR_PIXC_007_481_220L_20231210T141525_20231210T141536_PIC0_01.nc'), PosixPath('/data/scratch/globc/bonassies/data/SWOT/Ohio/SWOT_L2_HR_PIXC_007_466_088R_20231210T010139_20231210T010150_PGC0_01.nc'), PosixPath('/data/scratch/globc/bonassies/data/SWOT/Ohio/SWOT_L2_HR_PIXC_007_481_221L_20231210T141535_20231210T141546_PGC0_01.nc'), PosixPath('/data

	 SWOT_L2_HR_PIXC_003_481_220L_20230918T031508_20230918T031519_PGC0_01.nc
Specular Ringing: (array([False,  True]), array([5149160,   80771]))
Low Coherent: (array([False,  True]), array([5182951,   46980]))
Pixel discarded: (array([False,  True]), array([5225799,    4132]))


/data/home/globc/bonassies/.conda/envs/conda3.10/lib/python3.10/site-packages/xarray/core/computation.py:818: RuntimeWarning: invalid value encountered in sqrt
  result_data = func(*input_data)


	 SWOT_L2_HR_PIXC_003_481_221L_20230918T031518_20230918T031529_PGC0_01.nc
Specular Ringing: (array([False,  True]), array([4446981,  106892]))
Low Coherent: (array([False,  True]), array([4515240,   38633]))
Pixel discarded: (array([False,  True]), array([4550078,    3795]))


/data/home/globc/bonassies/.conda/envs/conda3.10/lib/python3.10/site-packages/xarray/core/computation.py:818: RuntimeWarning: invalid value encountered in sqrt
  result_data = func(*input_data)


...

/data/home/globc/bonassies/.conda/envs/conda3.10/lib/python3.10/site-packages/xarray/core/computation.py:818: RuntimeWarning: invalid value encountered in sqrt
  result_data = func(*input_data)
/data/home/globc/bonassies/.conda/envs/conda3.10/lib/python3.10/site-packages/xarray/core/computation.py:818: RuntimeWarning: divide by zero encountered in divide
  result_data = func(*input_data)


>>> Working on :
	 SWOT_L2_HR_PIXC_028_481_220L_20250220T180206_20250220T180217_PIC2_01.nc
Specular Ringing: (array([False,  True]), array([7715720,  232551]))
Low Coherent: (array([False,  True]), array([7805656,  142615]))
Pixel discarded: (array([False,  True]), array([7923459,   24812]))


/data/home/globc/bonassies/.conda/envs/conda3.10/lib/python3.10/site-packages/xarray/core/computation.py:818: RuntimeWarning: invalid value encountered in sqrt
  result_data = func(*input_data)


	 SWOT_L2_HR_PIXC_028_481_221L_20250220T180216_20250220T180227_PIC2_01.nc
Specular Ringing: (array([False,  True]), array([7539523,  324538]))
Low Coherent: (array([False,  True]), array([7703977,  160084]))
Pixel discarded: (array([False,  True]), array([7815757,   48304]))


/data/home/globc/bonassies/.conda/envs/conda3.10/lib/python3.10/site-packages/xarray/core/computation.py:818: RuntimeWarning: invalid value encountered in sqrt
  result_data = func(*input_data)


In [7]:
swot_project.Rasterizer.make_space = False
swot_project.Rasterizer.gpkg_to_tiff()

>>> Converting the SWOT geopackage to tiff
>>> Generate tiff for every variables
Working on  /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Ohio/gpkg_combined/SWOT_epsg32616_20250424T081730.gpkg
>>> Generate tiff for  sig0
gdal_grid -a invdistnn:power=2:smoothing=1:radius=50:max_points=20:nodata=-9999 -txe 465824.0000002095 525844.562499841 -tye 4213272.500110996 4176822.500110473 -outsize 6003.0 3646.0 -zfield sig0 -of GTiff -ot Float32 /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Ohio/gpkg_combined/SWOT_epsg32616_20250424T081730.gpkg /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Ohio/rasters/sig0/SWOT_epsg32616_20250424T081730_sig0.tif --config GDAL_NUM_THREADS 10 --config GDAL_CACHEMAX 160000
Grid data type is "Float32"
Grid size = (6003 3646).
Corner coordinates = (465824.000000 4213272.500111)-(525844.562500 4176822.500110).
Grid cell size = (9.998428 -9.997257).
Source point count = 3989179.
Algorithm name: "invdistnn".
Option

ERROR 6: /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Ohio/rasters/SWOT_epsg32616_20230918T031518_combined.tif, band 1: SetColorTable() not supported for multi-sample TIFF files.



Processing file     1 of     8,  0.000% completed in 0 minutes.
Filename: /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Ohio/rasters/sig0/SWOT_epsg32616_20230918T031518_sig0.tif
File Size: 6003x3646x1
Pixel Size: 9.998428 x -9.997257
UL:(465824.000000,4213272.500111)   LR:(525844.562500,4176822.500110)
Copy 0,0,6003,3646 to 0,0,6003,3646.

Processing file     2 of     8, 12.500% completed in 0 minutes.
Filename: /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Ohio/rasters/coherent_power/SWOT_epsg32616_20230918T031518_coherent_power.tif
File Size: 6003x3646x1
Pixel Size: 9.998428 x -9.997257
UL:(465824.000000,4213272.500111)   LR:(525844.562500,4176822.500110)
Copy 0,0,6003,3646 to 0,0,6003,3646.

Processing file     3 of     8, 25.000% completed in 0 minutes.
Filename: /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Ohio/rasters/incidence/SWOT_epsg32616_20230918T031518_incidence.tif
File Size: 6003x3646x1
Pixel Size: 9.998428 x -9.9972

ERROR 6: /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Ohio/rasters/SWOT_epsg32616_20231029T204525_combined.tif, band 1: SetColorTable() not supported for multi-sample TIFF files.



Processing file     1 of     8,  0.000% completed in 0 minutes.
Filename: /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Ohio/rasters/sig0/SWOT_epsg32616_20231029T204525_sig0.tif
File Size: 6003x3646x1
Pixel Size: 9.998428 x -9.997257
UL:(465824.000000,4213272.500111)   LR:(525844.562500,4176822.500110)
Copy 0,0,6003,3646 to 0,0,6003,3646.

Processing file     2 of     8, 12.500% completed in 0 minutes.
Filename: /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Ohio/rasters/coherent_power/SWOT_epsg32616_20231029T204525_coherent_power.tif
File Size: 6003x3646x1
Pixel Size: 9.998428 x -9.997257
UL:(465824.000000,4213272.500111)   LR:(525844.562500,4176822.500110)
Copy 0,0,6003,3646 to 0,0,6003,3646.

Processing file     3 of     8, 25.000% completed in 0 minutes.
Filename: /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Ohio/rasters/incidence/SWOT_epsg32616_20231029T204525_incidence.tif
File Size: 6003x3646x1
Pixel Size: 9.998428 x -9.9972

...


Processing file     1 of     8,  0.000% completed in 0 minutes.
Filename: /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Ohio/rasters/sig0/SWOT_epsg32616_20250414T095512_sig0.tif
File Size: 6003x3646x1
Pixel Size: 9.998428 x -9.997257
UL:(465824.000000,4213272.500111)   LR:(525844.562500,4176822.500110)
Copy 0,0,6003,3646 to 0,0,6003,3646.

Processing file     2 of     8, 12.500% completed in 0 minutes.
Filename: /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Ohio/rasters/coherent_power/SWOT_epsg32616_20250414T095512_coherent_power.tif
File Size: 6003x3646x1
Pixel Size: 9.998428 x -9.997257
UL:(465824.000000,4213272.500111)   LR:(525844.562500,4176822.500110)
Copy 0,0,6003,3646 to 0,0,6003,3646.

Processing file     3 of     8, 25.000% completed in 0 minutes.
Filename: /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Ohio/rasters/incidence/SWOT_epsg32616_20250414T095512_incidence.tif
File Size: 6003x3646x1
Pixel Size: 9.998428 x -9.9972

ERROR 6: /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Ohio/rasters/SWOT_epsg32616_20250423T190324_combined.tif, band 1: SetColorTable() not supported for multi-sample TIFF files.



Processing file     1 of     8,  0.000% completed in 0 minutes.
Filename: /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Ohio/rasters/sig0/SWOT_epsg32616_20250423T190324_sig0.tif
File Size: 6003x3646x1
Pixel Size: 9.998428 x -9.997257
UL:(465824.000000,4213272.500111)   LR:(525844.562500,4176822.500110)
Copy 0,0,6003,3646 to 0,0,6003,3646.

Processing file     2 of     8, 12.500% completed in 0 minutes.
Filename: /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Ohio/rasters/coherent_power/SWOT_epsg32616_20250423T190324_coherent_power.tif
File Size: 6003x3646x1
Pixel Size: 9.998428 x -9.997257
UL:(465824.000000,4213272.500111)   LR:(525844.562500,4176822.500110)
Copy 0,0,6003,3646 to 0,0,6003,3646.

Processing file     3 of     8, 25.000% completed in 0 minutes.
Filename: /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Ohio/rasters/incidence/SWOT_epsg32616_20250423T190324_incidence.tif
File Size: 6003x3646x1
Pixel Size: 9.998428 x -9.9972

ERROR 6: /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Ohio/rasters/SWOT_epsg32616_20250424T081730_combined.tif, band 1: SetColorTable() not supported for multi-sample TIFF files.



Processing file     1 of     8,  0.000% completed in 0 minutes.
Filename: /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Ohio/rasters/sig0/SWOT_epsg32616_20250424T081730_sig0.tif
File Size: 6003x3646x1
Pixel Size: 9.998428 x -9.997257
UL:(465824.000000,4213272.500111)   LR:(525844.562500,4176822.500110)
Copy 0,0,6003,3646 to 0,0,6003,3646.

Processing file     2 of     8, 12.500% completed in 0 minutes.
Filename: /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Ohio/rasters/coherent_power/SWOT_epsg32616_20250424T081730_coherent_power.tif
File Size: 6003x3646x1
Pixel Size: 9.998428 x -9.997257
UL:(465824.000000,4213272.500111)   LR:(525844.562500,4176822.500110)
Copy 0,0,6003,3646 to 0,0,6003,3646.

Processing file     3 of     8, 25.000% completed in 0 minutes.
Filename: /data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Ohio/rasters/incidence/SWOT_epsg32616_20250424T081730_incidence.tif
File Size: 6003x3646x1
Pixel Size: 9.998428 x -9.9972

If needed, we can put extra rasters to the same resolution and extent as the SWOT data. This is useful to compare the SWOT data with other data.

It uses gdal to rasterize the data. gdal is used as command line using os.system.

In [ ]:
# raster = Path("/data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Ohio/aux_data/ESA_WC_Fusion_cut_32616.tif")
# raster_crs = '32616'
# interp = 'near'
# swot_project.Rasterizer.gdalwarp_raster_to_swot_bbox_and_size(raster, raster_crs, interp)
# raster = Path("/data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Ohio/aux_data/FABDEM_Fusion_cut_32616.tif")
# raster_crs = '32616'
# interp = 'bilinear'
# swot_project.Rasterizer.gdalwarp_raster_to_swot_bbox_and_size(raster, raster_crs, interp)
# raster = Path("/data/scratch/globc/bonassies/workspace/swot_for_flood/examples/Ohio/aux_data/FM_20250222T000000_S1_POST_fusion_cut_32616.tif")
# raster_crs = '32616'
# interp = 'near'
# swot_project.Rasterizer.gdalwarp_raster_to_swot_bbox_and_size(raster, raster_crs, interp)

You can check other notebooks for more information about the library.